In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F
import numpy as np

In [ ]:
#################################
# Path
#################################
dataset_npz_path = "C:/Users/user/Desktop/IDS_masters/dataset/carchallenge_training_6channel.npz"
test_path = "C:/Users/user/Desktop/IDS_masters/dataset/carhacking_test_6channel.npz"

save_path1 = "cnn_model1.pth"
save_path2 = "cnn_model2.pth"

In [ ]:
#################################
# 1. Dataset Load Class
#################################

class LoadDatset(Dataset):
    def __init__(self, tensor_X, tensor_y):
        # [수정] NumPy 배열이 들어올 경우를 대비한 변환 로직
        if isinstance(tensor_X, np.ndarray):
            tensor_X = torch.from_numpy(tensor_X).float()
        if isinstance(tensor_y, np.ndarray):
            tensor_y = torch.from_numpy(tensor_y).long()

        # [추가] NaN(결측치)을 0.0으로 안전하게 대체하여 학습 붕괴 방지
        self.X = torch.nan_to_num(tensor_X, nan=0.0)
        self.y = tensor_y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
#################################
# 2. First Model
#################################
class First_cnn(nn.Module):
    def __init__(self, feat_ch, num_classes):
        super().__init__()

        in_ch = feat_ch

        self.conv = nn.Sequential(
            nn.Conv1d(in_ch,32,kernel_size=3,padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.AvgPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3,padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AvgPool1d(2),

            nn.AdaptiveAvgPool1d(1),
        )


        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64,num_classes)
        )


    def forward(self,x):
        h = self.conv(x)
        out = self.classifier(h)

        return out

In [ ]:
#################################
# 3. Second Model
#################################
class Second_cnn(nn.Module):
    def __init__(self, feat_ch, num_classes):
        super().__init__()

        in_ch = feat_ch

        self.conv = nn.Sequential(
            nn.Conv1d(in_ch,32,kernel_size=3,padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.AvgPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3,padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AvgPool1d(2),

            nn.AdaptiveAvgPool1d(1),
        )


        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64,num_classes)
        )


    def forward(self,x):
        h = self.conv(x)
        out = self.classifier(h)

        return out

In [ ]:
Config = {
    "NUM_CLASSE1": 2,         # 0 normal, 1 dos, 2 fuzzing, 3 replay, 4 spoofing
    "NUM_CLASSE2": 4,
    "BATCH_SIZE": 64,
    "EPOCHS": 10,
    "LR": 1e-3,
    "WEIGHT_DECAY": 1e-4,
    "SEED": 42,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

generator = torch.Generator()
generator.manual_seed(Config["SEED"])


data_np = np.load(dataset_npz_path)
X_np, y_np = data_np["X"], data_np["y"]

full_dataset = LoadDatset(X_np, y_np)

total_size = len(full_dataset)
train_size = int(0.8 * total_size)
val_size = total_size - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

train_loader = DataLoader(train_dataset, Config["BATCH_SIZE"], shuffle=True)
val_loader = DataLoader(val_dataset, Config["BATCH_SIZE"], shuffle=False)

In [ ]:
#################################
# 4. First Training
#################################

feat_ch = 6
model1 = First_cnn(
    feat_ch=feat_ch,
    num_classes = Config["NUM_CLASSE1"]
   ).to(device)

normal_label = 0

criterion1 = nn.CrossEntropyLoss()
optimizer1 = torch.optim.Adam(model1.parameters(), lr=0.001, weight_decay=Config["WEIGHT_DECAY"])

for epoch in range(20):  # loop over the dataset multiple times
    model1.train()
    running_loss = 0.0
    tr_total = 0
    
    for x,y in train_loader:
        x, y = x.to(device), y.to(device)

        y_bin = (y != normal_label).long()

        optimizer1.zero_grad()
        logits = model1(x)
        loss = criterion1(logits, y_bin)
        loss.backward()
        optimizer1.step()

        running_loss += loss.item() * y.size(0)
        tr_total +=y.size(0)
        tr_loss = running_loss / tr_total

    print(f'epoch: {epoch + 1} | loss: {tr_loss}')
    running_loss = 0.0

print('Finished Training')

torch.save(model1.state_dict(),save_path1)
print(f"Model saved to {save_path1}")

In [ ]:
#################################
# 5. Second Training
#################################

feat_ch = 1

model2 = Second_cnn(
    feat_ch=feat_ch,
    num_classes = Config["NUM_CLASSE2"]
   ).to(device)

criterion2 = nn.CrossEntropyLoss()
optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.001, weight_decay=Config["WEIGHT_DECAY"])

for epoch in range(20):  # loop over the dataset multiple times
    model2.train()
    running_loss = 0.0
    tr_total = 0

    for x,y in train_loader:
        x, y = x.to(device), y.to(device)

        mask = (y != normal_label)
        if mask.sum().item() == 0:
            continue

        x_a = x[mask]
        y_a = y[mask]

        y_4 = y_a - 1


        optimizer2.zero_grad()
        logits = model2(x_a)
        loss = criterion2(logits, y_4)
        loss.backward()
        optimizer2.step()

        running_loss += loss.item() * y_4.size(0)
        tr_total += y_4.size(0)
        tr_loss = running_loss / tr_total

    print(f'epoch: {epoch + 1} | loss: {tr_loss}')
    running_loss = 0.0

print('Finished Training')

torch.save(model2.state_dict(),save_path2)
print(f"Model saved to {save_path2}")

In [ ]:
#################################
# 6. CNN Test
#################################

normal_label = 0

def predict_to_stage(x, model1, model2, device, T):

    model1.eval()
    model2.eval()
    
    x = x.to(device)

    with torch.no_grad():
        logit1 = model1(x)
        p1 = F.softmax(logit1, dim=1)
        p_normal = p1[:,0]

        pred_final = torch.empty((x.size(0),), dtype=torch.long, device=x.device)
        normal_mask = (p_normal >= T)

        ##### normal ######
        normal_mask = (p_normal >= T)
        pred_final[normal_mask] = 0

        attack_mask = ~normal_mask
        if attack_mask.any():
            logit2 = model2(x[attack_mask])
            pred4 = logit2.argmax(dim=1)      # 0~3
            pred_final[attack_mask]  = pred4 + 1 # 1~4로 복원

        return pred_final

In [ ]:
num_classes = 5
conf_mat = torch.zeros(num_classes, num_classes, device='cpu')

LABEL_NAME = {0:"Normal", 1: "DoS", 2:"Fuzzing", 3:"Replay", 4:"Spoofing"}

test_data = np.load(test_path)
X_np, y_np = test_data["X"], test_data["y"]
test_dataset = LoadDatset(X_np, y_np)
test_loader = DataLoader(test_dataset, Config["BATCH_SIZE"], shuffle=False)

with torch.no_grad():
    for data in test_loader:
        x,y = data
        x,y = x.to(device), y.to(device)

        # 2단계 예측
        pred = predict_to_stage(x.permute(0,2,1), model1, model2, device=device, T=0.95)

        #### Accuracy ####
        total += y.size(0)
        correct += (pred == y).sum().item()

        #### confusion matrix ####
        y_cpu = y.detach().cpu().view(-1)
        pred_cpu = pred.detach().cpu().view(-1)

        for t, p in zip(y_cpu, pred_cpu):
            conf_mat[t.long(), p.long()] += 1


    accuracy = correct / max(total, 1)

tp = conf_mat.diag()
fp = conf_mat.sum(0) - tp
fn = conf_mat.sum(1) - tp

precision_per_class = tp / (tp + fp + 1e-12)
recall_per_class    = tp / (tp + fn + 1e-12)
f1_per_class        = 2 * precision_per_class * recall_per_class / (precision_per_class + recall_per_class + 1e-12)

precision_macro = precision_per_class.mean().item()
recall_macro    = recall_per_class.mean().item()
f1_macro        = f1_per_class.mean().item()

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision(macro): {precision_macro:.4f}")
print(f"Recall(macro)   : {recall_macro:.4f}")
print(f"F1(macro)       : {f1_macro:.4f}")
print("Confusion Matrix:")
print(conf_mat.int())

# 클래스별 정확도(= recall)
row_sum = conf_mat.sum(dim=1)
class_acc = tp / (row_sum + 1e-12)

print("\n=== Per-class Accuracy (Recall) ===")
for i in range(num_classes):
    n = int(row_sum[i].item())
    acc_i = 100.0 * class_acc[i].item()
    name = LABEL_NAME.get(i, f"class_{i}")
    print(f"{name:>10s} : {acc_i:6.2f}%  (correct {int(tp[i].item())}/{n})")

